In [1]:
import re
import html
import nltk
import pandas as pd
import numpy as np
from nltk.sentiment import SentimentIntensityAnalyzer

In [2]:
nltk.download('vader_lexicon', quiet=True)




True

In [ ]:

print("Loading merged data...")
df = pd.read_csv("../data/processed/merged_raw.csv")
print(f"Loaded {len(df)} rows")


# CLEAN TEXT — vectorized 

print("\nP1: Cleaning text...")

def clean_series(series):
    """Clean an entire column at once using vectorized string ops"""
    s = series.fillna('')
    s = s.str.replace(r'<[^>]+>', ' ', regex=True)     # Remove HTML tags
    s = s.str.replace(r'http\S+', '', regex=True)       # Remove URLs
    s = s.str.replace(r'[^\w\s.,!?$%\'\-]', ' ', regex=True)
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()
    # Mark short results as empty
    s = s.where(s.str.split().str.len() >= 8, '')
    return s

# Clean both columns at once — no loop
df['review_text_clean']   = clean_series(df['review_text'])
df['product_name_clean']  = clean_series(df['product_name'])

# Drop rows where review became empty after cleaning
df = df[df['review_text_clean'] != ''].reset_index(drop=True)
print(f"After cleaning: {len(df)} rows")


# BUILD INPUT TEXT — vectorized (fast)

print("\nP2: Building input text...")

# String concatenation on whole column — no apply needed
df['input_text'] = (
    "PRODUCT: " + df['product_name_clean'] +
    ". REVIEW: "  + df['review_text_clean']
)

# Truncate to 350 words — vectorized
def truncate_series(series, max_words=350):
    return series.apply(lambda x: ' '.join(str(x).split()[:max_words]))

df['input_text'] = truncate_series(df['input_text'])
print("Input text built.")


#  FEATURE ENGINEERING — mixed (some vectorized, VADER batched)

print("\nP3: Engineering features...")

# Fast vectorized features
df['word_count']         = df['input_text'].str.split().str.len()
df['exclamation_count']  = df['input_text'].str.count('!')
df['question_count']     = df['input_text'].str.count(r'\?')

text_lower = df['input_text'].str.lower()  # Lowercase once, reuse

# Emotional wordsSystem 1 
S1_WORDS = ['love', 'amazing', 'perfect', 'obsessed', 'adore', 'gorgeous',
            'wonderful', 'excited', 'happy', 'cute', 'fun', 'favorite',
            'impulse', 'spontaneous', 'gift', 'treat', 'splurge']

# Rational words (System 2 
S2_WORDS = ['research', 'compared', 'specification', 'warranty', 'technical',
            'performance', 'durable', 'efficient', 'professional', 'compatible',
            'accurate', 'reliable', 'features', 'reviewed', 'battery life']

# Count keyword hits per row — vectorized using regex OR pattern
s1_pattern = '|'.join(S1_WORDS)
s2_pattern = '|'.join(S2_WORDS)

df['s1_keyword_hits'] = text_lower.str.count(s1_pattern)
df['s2_keyword_hits'] = text_lower.str.count(s2_pattern)

df['emotional_ratio'] = (df['s1_keyword_hits'] / df['word_count'].clip(lower=1)).round(4)
df['rational_ratio']  = (df['s2_keyword_hits'] / df['word_count'].clip(lower=1)).round(4)

# Star rating 
df['star_rating_clean'] = pd.to_numeric(df.get('star_rating', 3), errors='coerce').fillna(3)

print("Vectorized features done.")

#  VADER sentiment — batched 
print("Running VADER sentiment (batched)...")
#
sia = SentimentIntensityAnalyzer()

# Process in batches of 500 — much faster than one-by-one
BATCH_SIZE = 500
texts = df['input_text'].tolist()

compound_scores = []
pos_scores      = []
neg_scores      = []

total = len(texts)
for i in range(0, total, BATCH_SIZE):
    batch = texts[i : i + BATCH_SIZE]
    for text in batch:
        scores = sia.polarity_scores(text)
        compound_scores.append(scores['compound'])
        pos_scores.append(scores['pos'])
        neg_scores.append(scores['neg'])

    # Progress update every batch
    done = min(i + BATCH_SIZE, total)
    print(f"  Sentiment: {done}/{total} rows done", end='\r')

df['sentiment_compound'] = compound_scores
df['sentiment_pos']      = pos_scores
df['sentiment_neg']      = neg_scores
print(f"\nSentiment done for {total} rows.")


# P4: BALANCE CLASSES

print("\nP4: Balancing classes...")
#
s1_count = (df.cognitive_label == 1).sum()
s2_count = (df.cognitive_label == 0).sum()
target   = min(s1_count, s2_count, 3000)

df_s1 = df[df.cognitive_label == 1].sample(target, random_state=42)
df_s2 = df[df.cognitive_label == 0].sample(target, random_state=42)

df_balanced = pd.concat([df_s1, df_s2]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print(f"Balanced: {len(df_balanced)} rows | S1={target} | S2={target}")
#
df_balanced.to_csv("../data/processed/full_dataset_preprocessed.csv", index=False)
print("Saved → ../data/processed/full_dataset_preprocessed.csv")


# P5: TRAIN / VAL / TEST SPLIT

print("\nP5: Splitting dataset...")

from sklearn.model_selection import train_test_split

KEEP_COLS = ['input_text', 'cognitive_label', 'product_name_clean',
             'category', 'source_file', 'word_count', 'sentiment_compound',
             'emotional_ratio', 'rational_ratio', 'star_rating_clean',
             'exclamation_count']

df_out = df_balanced[[c for c in KEEP_COLS if c in df_balanced.columns]]

train_val, test = train_test_split(
    df_out, test_size=0.15,
    stratify=df_out['cognitive_label'], random_state=42
)
train, val = train_test_split(
    train_val, test_size=0.1765,
    stratify=train_val['cognitive_label'], random_state=42
)

train.to_csv("../data/final/train.csv", index=False)
val.to_csv("../data/final/val.csv",     index=False)
test.to_csv("../data/final/test.csv",   index=False)

print(f"\n{'='*45}")
print("STEP 0 COMPLETE")
print(f"{'='*45}")
print(f"Train : {len(train):,} rows")
print(f"Val   : {len(val):,} rows")
print(f"Test  : {len(test):,} rows")
print(f"{'='*45}")

Loading merged data...
Loaded 11600 rows

P1: Cleaning text...
After cleaning: 10791 rows

P2: Building input text...
Input text built.

P3: Engineering features...
Vectorized features done.
Running VADER sentiment (batched)...
  Sentiment: 10791/10791 rows done
Sentiment done for 10791 rows.

P4: Balancing classes...
Balanced: 5178 rows | S1=2589 | S2=2589
Saved → ../data/processed/full_dataset_preprocessed.csv

P5: Splitting dataset...

STEP 0 COMPLETE
Train : 3,624 rows
Val   : 777 rows
Test  : 777 rows


In [5]:
# Sanity check before Step 1
train = pd.read_csv("../data/final/train.csv")
val   = pd.read_csv("../data/final/val.csv")
test  = pd.read_csv("../data/final/test.csv")

print("="*50)
print("STEP 0 COMPLETE — DATASET QUALITY REPORT")
print("="*50)
for name, df in [("Train", train), ("Val", val), ("Test", test)]:
    s1 = (df.cognitive_label == 1).sum()
    s2 = (df.cognitive_label == 0).sum()
    print(f"\n{name}: {len(df)} rows | S1={s1} | S2={s2}")

print(f"\nAvg word count: {train['word_count'].mean():.0f}")
print(f"Avg sentiment:  {train['sentiment_compound'].mean():.3f}")
print(f"\nS1 avg sentiment: {train[train.cognitive_label==1]['sentiment_compound'].mean():.3f}")
print(f"S2 avg sentiment: {train[train.cognitive_label==0]['sentiment_compound'].mean():.3f}")
# S1 should be higher — validates your labeling is working
print("="*50)

STEP 0 COMPLETE — DATASET QUALITY REPORT

Train: 3624 rows | S1=1812 | S2=1812

Val: 777 rows | S1=389 | S2=388

Test: 777 rows | S1=388 | S2=389

Avg word count: 62
Avg sentiment:  0.374

S1 avg sentiment: 0.378
S2 avg sentiment: 0.370


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
train = pd.read_csv("../data/final/train.csv")
val   = pd.read_csv("../data/final/val.csv")
test  = pd.read_csv("../data/final/test.csv")
full  = pd.concat([train, val, test], ignore_index=True)

print("=" * 55)
print("   DATASET QUALITY & LABEL ACCURACY REPORT")
print("=" * 55)

# Class Balance 
s1 = (full.cognitive_label == 1).sum()
s2 = (full.cognitive_label == 0).sum()
ratio = min(s1, s2) / max(s1, s2)

print(f"\n[1] CLASS BALANCE")
print(f"    System 1 (emotional) : {s1:,} rows")
print(f"    System 2 (rational)  : {s2:,} rows")
print(f"    Balance ratio        : {ratio:.2f}")

if ratio >= 0.90:
    print("    STATUS → EXCELLENT (ratio ≥ 0.90)")
elif ratio >= 0.75:
    print("    STATUS → ACCEPTABLE (ratio ≥ 0.75)")
else:
    print("    STATUS → WARNING — classes are imbalanced, use class_weight in training")

# Check 2: Sentiment Separation

s1_sent = full[full.cognitive_label == 1]['sentiment_compound'].mean()
s2_sent = full[full.cognitive_label == 0]['sentiment_compound'].mean()
sent_gap = s1_sent - s2_sent

print(f"\n[2] SENTIMENT SEPARATION (label validation)")
print(f"    S1 avg sentiment : {s1_sent:.4f}")
print(f"    S2 avg sentiment : {s2_sent:.4f}")
print(f"    Gap (S1 - S2)    : {sent_gap:.4f}")

if sent_gap > 0.05:
    print("    STATUS → PASS — S1 is more positive than S2 as expected")
elif sent_gap > 0:
    print("    STATUS → WEAK — small gap, labels may be noisy")
else:
    print("    STATUS → FAIL — S2 is MORE positive than S1, check your labels")

# ── Check 3: Keyword Separation ──────────────────────────
s1_emo = full[full.cognitive_label == 1]['emotional_ratio'].mean()
s2_emo = full[full.cognitive_label == 0]['emotional_ratio'].mean()
s1_rat = full[full.cognitive_label == 1]['rational_ratio'].mean()
s2_rat = full[full.cognitive_label == 0]['rational_ratio'].mean()

print(f"\n[3] KEYWORD SEPARATION (language pattern check)")
print(f"    Emotional word ratio — S1: {s1_emo:.4f} | S2: {s2_emo:.4f}")
print(f"    Rational word ratio  — S1: {s1_rat:.4f} | S2: {s2_rat:.4f}")

emo_ok = s1_emo > s2_emo
rat_ok = s2_rat > s1_rat

print(f"    S1 uses more emotional words : {'PASS' if emo_ok else 'FAIL'}")
print(f"    S2 uses more rational words  : {'PASS' if rat_ok else 'FAIL'}")

# ── Check 4: Exclamation Mark Separation 
s1_exc = full[full.cognitive_label == 1]['exclamation_count'].mean()
s2_exc = full[full.cognitive_label == 0]['exclamation_count'].mean()

print(f"\n[4] EXCLAMATION MARKS (emotional energy signal)")
print(f"    S1 avg exclamations : {s1_exc:.3f}")
print(f"    S2 avg exclamations : {s2_exc:.3f}")
print(f"    STATUS → {'PASS' if s1_exc > s2_exc else 'FAIL'} "
      f"(S1 should have more !)")

# ── Check 5: Word Count Separation 
s1_wc = full[full.cognitive_label == 1]['word_count'].mean()
s2_wc = full[full.cognitive_label == 0]['word_count'].mean()

print(f"\n[5] WORD COUNT (review depth signal)")
print(f"    S1 avg word count : {s1_wc:.1f}")
print(f"    S2 avg word count : {s2_wc:.1f}")
print(f"    STATUS → {'PASS' if s2_wc > s1_wc else 'NOTE'} "
      f"(S2 tends to write longer, more detailed reviews)")

# ── Check 6: Source Distribution 
print(f"\n[6] SOURCE FILE DISTRIBUTION")
for src, count in full['source_file'].value_counts().items():
    pct = count / len(full) * 100
    label = "S1" if full[full.source_file==src]['cognitive_label'].mode()[0] == 1 else "S2"
    print(f"    [{label}] {src:<45} {count:>5} rows ({pct:.1f}%)")

# ── Check 7: Missing Values 
print(f"\n[7] MISSING VALUES")
critical_cols = ['input_text', 'cognitive_label', 'sentiment_compound',
                 'emotional_ratio', 'rational_ratio']
for col in critical_cols:
    missing = full[col].isna().sum()
    status = "OK" if missing == 0 else f"WARNING — {missing} missing"
    print(f"    {col:<25} : {status}")

# ── Check 8: Text Length Sanity 
too_short = (full['word_count'] < 10).sum()
too_long  = (full['word_count'] > 400).sum()

print(f"\n[8] TEXT LENGTH SANITY")
print(f"    Rows with < 10 words  : {too_short} "
      f"{'(OK)' if too_short == 0 else '(WARNING — remove these)'}")
print(f"    Rows with > 400 words : {too_long} "
      f"{'(OK)' if too_long == 0 else '(OK — will be truncated by tokenizer)'}")

# ── Overall Dataset Score
checks = [ratio >= 0.75, sent_gap > 0, emo_ok, rat_ok, s1_exc > s2_exc]
score = sum(checks)
print(f"\n{'='*55}")
print(f"   OVERALL DATASET SCORE: {score}/5 checks passed")
if score == 5:
    print("   STATUS → DATASET IS RELIABLE. Proceed to model training.")
elif score >= 3:
    print("   STATUS → DATASET IS USABLE. Review failed checks above.")
else:
    print("   STATUS → DATASET NEEDS FIXING before training.")
print(f"{'='*55}")

   DATASET QUALITY & LABEL ACCURACY REPORT

[1] CLASS BALANCE
    System 1 (emotional) : 2,589 rows
    System 2 (rational)  : 2,589 rows
    Balance ratio        : 1.00
    STATUS → EXCELLENT (ratio ≥ 0.90)

[2] SENTIMENT SEPARATION (label validation)
    S1 avg sentiment : 0.3772
    S2 avg sentiment : 0.3730
    Gap (S1 - S2)    : 0.0042
    STATUS → WEAK — small gap, labels may be noisy

[3] KEYWORD SEPARATION (language pattern check)
    Emotional word ratio — S1: 0.0106 | S2: 0.0061
    Rational word ratio  — S1: 0.0009 | S2: 0.0022
    S1 uses more emotional words : PASS
    S2 uses more rational words  : PASS

[4] EXCLAMATION MARKS (emotional energy signal)
    S1 avg exclamations : 0.456
    S2 avg exclamations : 0.334
    STATUS → PASS (S1 should have more !)

[5] WORD COUNT (review depth signal)
    S1 avg word count : 54.8
    S2 avg word count : 69.3
    STATUS → PASS (S2 tends to write longer, more detailed reviews)

[6] SOURCE FILE DISTRIBUTION
    [S2] amazon_reviews_us